<a href="https://colab.research.google.com/github/fernandoman665/semana03-Fundamentos-de-Gesti-n-de-Datos/blob/main/LAB_D3_FGD_2026II_QuispeJaraFernandoManuel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Lab03_Fundamentos de Gestion de DATOS

Caso de estudio:

TecnoAndina Retail es una cadena peruana de tecnología con tiendas en Lima, Arequipa, Trujillo y Piura y un canal de venta por Web. Su información está repartida en cuatro tablas y el área comercial necesita saber qué clientes compran más, cuáles todavía no compran y qué productos y categorías sustentan las ventas.

Naturaleza de los datos. Conjunto de práctica inventado (no son ventas ni personas reales), con pedidos del 1 de agosto al 30 de septiembre de 2026, publicado en GitHub en LD3_TecnoAndina_Retail.xlsx. Sirve para practicar consultas, no para afirmar tendencias ni rentabilidad.

•  clientes: id_cliente, nombre, ciudad. Una fila es un cliente registrado.

•  productos: id_producto, nombre_producto, categoria, precio_lista. Una fila es un producto del catálogo.

•  pedidos: id_pedido, id_cliente, fecha_pedido, canal, total. Una fila es un pedido de un cliente.

•  detalles_pedido: id_detalle, id_pedido, id_producto, cantidad, precio_unitario. Una fila es un producto incluido en un pedido.



In [18]:
import sqlite3
import pandas as pd

# Base de práctica de TecnoAndina Retail (agosto y septiembre de 2026)
url = "https://raw.githubusercontent.com/Rociosayan/Repo_FDG_2026-2/main/semana03/datos/LD3_TecnoAndina_Retail.xlsx"
conexion = sqlite3.connect(":memory:")

# Cada hoja del Excel pasa a una tabla de SQLite
pd.read_excel(url, sheet_name="clientes").to_sql("clientes", conexion, index=False)
pd.read_excel(url, sheet_name="productos").to_sql("productos", conexion, index=False)
pd.read_excel(url, sheet_name="pedidos").to_sql("pedidos", conexion, index=False)
pd.read_excel(url, sheet_name="detalles_pedido").to_sql("detalles_pedido", conexion, index=False)

consulta = """SELECT
  (SELECT COUNT(*) FROM clientes) AS clientes,
  (SELECT COUNT(*) FROM productos) AS productos,
  (SELECT COUNT(*) FROM pedidos) AS pedidos,
  (SELECT COUNT(*) FROM detalles_pedido) AS detalles_pedido;"""
pd.read_sql_query(consulta, conexion)


,clientes,productos,pedidos,detalles_pedido
0,24,14,42,68


In [19]:
consulta = """SELECT COUNT(p.id_pedido) AS pedidos_unidos,
       COUNT(DISTINCT c.id_cliente) AS clientes_con_pedidos,
       (SELECT COUNT(*) FROM clientes) AS clientes_registrados
FROM pedidos AS p
INNER JOIN clientes AS c ON p.id_cliente = c.id_cliente;"""
pd.read_sql_query(consulta, conexion)


,pedidos_unidos,clientes_con_pedidos,clientes_registrados
0,42,18,24


In [20]:
consulta = """SELECT c.id_cliente, c.nombre, c.ciudad
FROM clientes AS c
LEFT JOIN pedidos AS p ON c.id_cliente = p.id_cliente
WHERE p.id_pedido IS NULL
ORDER BY c.id_cliente;"""
pd.read_sql_query(consulta, conexion)


,id_cliente,nombre,ciudad
0,6,Andrés Villanueva,Lima
1,11,Raúl Benavides,Arequipa
2,15,Héctor Burga,Trujillo
3,20,Marta Infante,Piura
4,22,Gabriela Ponce,Arequipa
5,24,Natalia Ato,Piura


# Pregunta 1. ¿Cuántos pedidos y cuántos clientes distintos devuelve la celda 2? ¿Por qué INNER JOIN no muestra a todos los clientes registrados?

Nos devuelve 42 pedidos y 18 clientes, y la funcion INNER JOIN, no muestra la totalidad de clientes porque no todos los registros de los clientes estan conectados.

# Pregunta 2. En la celda 3, ¿qué tabla es la izquierda y por qué se filtra con p.id_pedido IS NULL y no con = NULL? ¿Cuántos clientes no tienen pedidos?

En una base de datos, los valores nulos no pueden evaluarse, por eso se usa el operador IS NULL, esto nos ayuda a que se tenga una respuesta, en este caso nos da los 6 clientes que no registran pedidos.




In [21]:
#RIGHT JOIN y FULL OUTER JOIN
#Se solicita. Ejecute la celda 4 (versión y RIGHT JOIN)
#y la celda 5, que crea un resumen por cliente para cada mes y los une con FULL OUTER JOIN.

# RIGHT JOIN y FULL OUTER JOIN requieren SQLite 3.39.0 o posterior
consulta = """SELECT sqlite_version() AS version_sqlite;"""
display(pd.read_sql_query(consulta, conexion))

consulta = """SELECT pr.id_producto, pr.nombre_producto, pr.categoria
FROM detalles_pedido AS d
RIGHT JOIN productos AS pr ON d.id_producto = pr.id_producto
WHERE d.id_detalle IS NULL
ORDER BY pr.id_producto;"""
display(pd.read_sql_query(consulta, conexion))




,version_sqlite
0,3.45.1


,id_producto,nombre_producto,categoria
0,5,Impresora multifuncional,Cómputo
1,14,Reloj inteligente,Accesorios


In [22]:
conexion.executescript("""
DROP VIEW IF EXISTS resumen_agosto;
DROP VIEW IF EXISTS resumen_septiembre;
CREATE TEMP VIEW resumen_agosto AS
SELECT id_cliente, COUNT(id_pedido) AS num_pedidos, SUM(total) AS monto
FROM pedidos
WHERE fecha_pedido BETWEEN '2026-08-01' AND '2026-08-31'
GROUP BY id_cliente;
CREATE TEMP VIEW resumen_septiembre AS
SELECT id_cliente, COUNT(id_pedido) AS num_pedidos, SUM(total) AS monto
FROM pedidos
WHERE fecha_pedido BETWEEN '2026-09-01' AND '2026-09-30'
GROUP BY id_cliente;
""")
consulta = """SELECT COALESCE(a.id_cliente, s.id_cliente) AS id_cliente,
       a.monto AS monto_agosto, s.monto AS monto_septiembre
FROM resumen_agosto AS a
FULL OUTER JOIN resumen_septiembre AS s ON a.id_cliente = s.id_cliente
ORDER BY id_cliente;"""
pd.read_sql_query(consulta, conexion)


,id_cliente,monto_agosto,monto_septiembre
0,1,1435.0,1967.0
1,2,1522.0,2615.0
2,3,656.0,840.0
3,4,2452.0,NaN
4,5,NaN,640.0
5,7,165.0,3460.0
6,8,271.0,236.0
7,9,220.0,NaN
8,10,6400.0,723.0
9,12,NaN,1420.0


# Pregunta 3. ¿Qué versión de SQLite informó la celda 4? En el RIGHT JOIN, ¿qué tabla es la derecha y qué productos no se vendieron?

La version SQLite es 3.45.1
WHERE d.id_detalle IS NULL; Los productos que no se vendieron son la impresora multifuncional y el reloj inteligente,

# Pregunta 4. En la celda 5, ¿cuántos clientes compraron solo en agosto, solo en septiembre y en ambos meses? ¿Cómo los reconoce por los NULL?

Se reconoce por el NULL, cuando al cruzar Agosto y setiembre nos manda un (NaN), es decir si EN Agosto tenemos NaN, son los que compraron en Setiembre, y si en la columna Setiembre tenenmos NaN, se entiende que compraron en Agosto, y si en ambas columnas no hay null en las filas comparativas, se compraron en ambos meses.


In [23]:
#Ejercicio 3 — WHERE, GROUP BY y HAVING
#La jefatura pide los clientes con al menos dos pedidos en septiembre.
#WHERE filtra filas antes de agrupar; GROUP BY forma un grupo por cliente; HAVING filtra los grupos ya formados.
consulta = """SELECT c.id_cliente, c.nombre,
       COUNT(p.id_pedido) AS pedidos_septiembre,
       SUM(p.total) AS monto_septiembre
FROM clientes AS c
INNER JOIN pedidos AS p ON c.id_cliente = p.id_cliente
WHERE p.fecha_pedido >= '2026-09-01'
GROUP BY c.id_cliente, c.nombre
HAVING COUNT(p.id_pedido) >= 2
ORDER BY monto_septiembre DESC, c.id_cliente;"""
pd.read_sql_query(consulta, conexion)


,id_cliente,nombre,pedidos_septiembre,monto_septiembre
0,2,Carlos Mendoza,3,2615
1,1,Lucía Paredes,2,1967
2,12,Sofía Linares,2,1420
3,17,Luis Távara,2,885
4,10,Patricia Zevallos,2,723


# Pregunta 5. ¿Qué filtra WHERE y qué filtra HAVING?
*Un cliente con tres pedidos en total, pero uno solo en septiembre, ¿aparece? Justifique.*

RWHERE filtra los pedidos realizados por fecha antes de agruparlos

HAVING filtra los clientes de acuerdo a su indicacion aqui es >=2.

No aparece ese cliente ya que solo tiene una compra en Septiembre, y segun el filtro HAVING, no cumple el requisito.


In [24]:
#Ejercicio 4 — Subconsulta: pedidos sobre el promedio por pedido
#Logística quiere revisar los pedidos que superan el monto de un pedido promedio. La subconsulta calcula primero el promedio; la consulta exterior compara cada pedido con ese valor.

#Se solicita. Ejecute la celda 7.
consulta = """SELECT p.id_pedido, p.fecha_pedido, c.id_cliente, c.nombre, p.total
FROM pedidos AS p
INNER JOIN clientes AS c ON p.id_cliente = c.id_cliente
WHERE p.total > (SELECT AVG(total) FROM pedidos)
ORDER BY p.total DESC, p.id_pedido;"""
pd.read_sql_query(consulta, conexion)

,id_pedido,fecha_pedido,id_cliente,nombre,total
0,3,2026-08-02,10,Patricia Zevallos,6400
1,13,2026-08-17,13,Diego Alva,5655
2,26,2026-09-08,7,Carlos Mendoza,3460
3,31,2026-09-15,18,Ana Girón,3045
4,35,2026-09-24,2,Carlos Mendoza,2544
5,20,2026-08-21,4,Jorge Salazar,2452
6,9,2026-08-12,16,Camila Neyra,2256
7,15,2026-08-18,17,Luis Távara,1680
8,11,2026-08-16,2,Carlos Mendoza,1522
9,28,2026-09-10,1,Lucía Paredes,1485


# Pregunta 7. ¿Qué devuelve la subconsulta y cuántos pedidos la superan? ¿Qué clientes aparecen más de una vez?

Nos devuelve el promedio global de todos los pedidos , si los registros superan dicho promedio, se tienen a los clientes que matematicamente son superiores al promedio, en este caso son 13 tickets que aparecen más, y los clientes que mas se repiten son: Carlos Mendoza y Lucia Paredes.

In [25]:
#Ejercicio 5 — Vista: guardar el resumen por cliente
#Varias áreas piden el mismo resumen por cliente. Una vista guarda la consulta con un nombre y se vuelve a ejecutar cada vez que se consulta.

#Se solicita. Ejecute la celda 8, que crea vista_resumen_cliente y la consulta.
conexion.executescript("""
DROP VIEW IF EXISTS vista_resumen_cliente;
CREATE VIEW vista_resumen_cliente AS
SELECT c.id_cliente, c.nombre, c.ciudad,
       COUNT(p.id_pedido) AS num_pedidos,
       COALESCE(SUM(p.total), 0) AS monto_acumulado,
       ROUND(AVG(p.total), 2) AS promedio_pedido
FROM clientes AS c
LEFT JOIN pedidos AS p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente, c.nombre, c.ciudad;
""")
consulta = """SELECT *
FROM vista_resumen_cliente
WHERE nombre = 'Carlos Mendoza' OR ciudad = 'Piura'
ORDER BY ciudad, monto_acumulado DESC, id_cliente;"""
pd.read_sql_query(consulta, conexion)

,id_cliente,nombre,ciudad,num_pedidos,monto_acumulado,promedio_pedido
0,7,Carlos Mendoza,Arequipa,2,3625,1812.50
1,2,Carlos Mendoza,Lima,4,4137,1034.25
2,17,Luis Távara,Piura,4,3224,806.00
3,18,Ana Girón,Piura,1,3045,3045.00
4,19,Pedro Seminario,Piura,2,1502,751.00
5,20,Marta Infante,Piura,0,0,NaN
6,24,Natalia Ato,Piura,0,0,NaN


# Pregunta 8. ¿Qué guarda la vista: los datos o la consulta? ¿Qué columna distingue a los dos clientes llamados Carlos Mendoza?

La vista se encarga de guardar unicamente las instrucciones de consulta, es decir son lineas de codigo a ejecutarse unicamente cuando se invoca.

En cambio el id_cliente se encarga de distinguir a dos o mas tipos de clientes, en este caso "Carlos Mendoza"


In [26]:
#Actividad 3 — Informe analítico comparado
#La gerencia evalúa ofrecer beneficios a los clientes que más compraron. Ejecute la celda 9, complete la tabla y responda citando cifras. Liderar por monto en dos meses no convierte a un cliente en el más rentable.
# Clientes con mayor monto acumulado (top cinco)
consulta = """SELECT c.id_cliente, c.nombre, c.ciudad,
       COUNT(p.id_pedido) AS num_pedidos,
       SUM(p.total) AS monto_acumulado,
       ROUND(AVG(p.total), 2) AS promedio_pedido
FROM clientes AS c
INNER JOIN pedidos AS p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente, c.nombre, c.ciudad
ORDER BY monto_acumulado DESC, c.id_cliente
LIMIT 5;"""
pd.read_sql_query(consulta, conexion)

,id_cliente,nombre,ciudad,num_pedidos,monto_acumulado,promedio_pedido
0,10,Patricia Zevallos,Arequipa,3,7123,2374.33
1,13,Diego Alva,Trujillo,2,5879,2939.50
2,2,Carlos Mendoza,Lima,4,4137,1034.25
3,7,Carlos Mendoza,Arequipa,2,3625,1812.50
4,1,Lucía Paredes,Lima,4,3402,850.50


#¿Qué clientes lideran por monto acumulado y quién tiene el mayor promedio por pedido?

Patricia Zevallos y Diego Alva lideran el mayor promedio en monto acumulado (7123 Y 5879) y el de mayor promedio por pedidolo tiene Diego alva con 2939.50

#¿Qué información adicional pediría antes de recomendar beneficios? Redacte una recomendación prudente en tres líneas.

Analizando los margenes de ganancia por categoria y producto, se recomienda dar enfazis a los mejores clientes, con campañas de fidelizacion para asi poder elevar sus frecuencias de consumo.

# Reto de aplicación y retroalimentación

# Indicaciones. Cada reto presenta la consulta ya escrita: ejecútela, identifique qué hace cada cláusula e interprete el resultado en no más de cuatro líneas. Los retos son la base de la actividad calificada de la semana.

Reto 1 — Clientes sin pedidos por ciudad
Marketing quiere saber en qué ciudades se concentran los clientes que todavía no tienen pedidos. Un cliente sin pedidos no es necesariamente un cliente perdido.

In [27]:
consulta = """SELECT c.ciudad, COUNT(c.id_cliente) AS clientes_sin_pedidos
FROM clientes AS c
LEFT JOIN pedidos AS p ON c.id_cliente = p.id_cliente
WHERE p.id_pedido IS NULL
GROUP BY c.ciudad
ORDER BY clientes_sin_pedidos DESC, c.ciudad;"""
pd.read_sql_query(consulta, conexion)

,ciudad,clientes_sin_pedidos
0,Arequipa,2
1,Piura,2
2,Lima,1
3,Trujillo,1


*¿Cuántos clientes sin pedidos hay y en qué ciudades se concentran? ¿Qué cláusula los conserva y cuál los aísla?*

Se tiene 6 clientes sin pedidos: en Arequipa son 2 , en Piura 2,Lima 1 y en Trujillo 1.

LEFT JOIN nos permite conserva a los clientes
WHERE filtra a los que no tienen pedido.

# Reto 2 — Ingresos por categoría con un umbral
# Finanzas quiere saber qué categorías generaron al menos S/ 10 000 entre agosto y septiembre. El ingreso se calcula desde las líneas de detalle.

In [28]:
consulta = """SELECT pr.categoria,
       SUM(d.cantidad) AS unidades,
       SUM(d.cantidad * d.precio_unitario) AS ingreso
FROM productos AS pr
INNER JOIN detalles_pedido AS d ON pr.id_producto = d.id_producto
GROUP BY pr.categoria
HAVING SUM(d.cantidad * d.precio_unitario) >= 10000
ORDER BY ingreso DESC, pr.categoria;"""
pd.read_sql_query(consulta, conexion)

,categoria,unidades,ingreso
0,Celulares,12,18805
1,Cómputo,37,17662


*Pregunta de cierre. ¿Qué categorías superan el umbral y cuál queda fuera? ¿Por qué la condición va en HAVING y no en WHERE?*

Las categorias que superan el umbral son celulares y cónputo.
La categoria accesorios queda fuera.

Se utiliza la condicion HAVING porque al ser ingreso total es un resultado matematico

# Reto 3 — Clientes sobre el promedio por cliente
# Comercial quiere identificar a los clientes cuyo monto acumulado supera el promedio de los clientes que compraron. La consulta usa la vista de la celda 8 y una subconsulta.

In [29]:
consulta = """SELECT id_cliente, nombre, ciudad, monto_acumulado
FROM vista_resumen_cliente
WHERE monto_acumulado > (SELECT AVG(monto_acumulado)
                         FROM vista_resumen_cliente
                         WHERE num_pedidos > 0)
ORDER BY monto_acumulado DESC, id_cliente;"""
pd.read_sql_query(consulta, conexion)

,id_cliente,nombre,ciudad,monto_acumulado
0,10,Patricia Zevallos,Arequipa,7123
1,13,Diego Alva,Trujillo,5879
2,2,Carlos Mendoza,Lima,4137
3,7,Carlos Mendoza,Arequipa,3625
4,1,Lucía Paredes,Lima,3402
5,17,Luis Távara,Piura,3224
6,18,Ana Girón,Piura,3045
7,16,Camila Neyra,Trujillo,2518
8,4,Jorge Salazar,Lima,2452


# Pregunta de cierre.

*¿Qué valor devuelve la subconsulta y cuántos clientes lo superan? ¿Por qué no debe compararse con el promedio por pedido del ejercicio 4?*

La subconsulta devuelve el promedio del monto acumulado de los clientes , se tiene 9 clientes que lo superan.

No se compara con el promedio del ejercicio 4 porque este utiliza transacciones individuales (por cliente) y no el acumulado

# Conclusiones

*Escriba tres conclusiones técnicas y de negocio de este laboratorio.*

// El uso correcto de los operadores INNER JOIN y LEFT JOIN. //

// GROUP BY, WHERE Y HAVING, nos ayuda de una manera significativa al momento de realizar consultas en SQLite, evitando errores. //


// Aplicar Tecnicas Estadisticas nos ayuda en la toma de deciciones y tambien en la busqueda de resultados para poder asi tener una mejor vision de negocio y una mejora significativa en nuestros resultados generales //